In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:
from confluent_kafka.admin import AdminClient

admin = AdminClient(conf)
md = admin.list_topics(timeout=10)       # fetches cluster metadata

print("Connected to the cluster.")
print("Brokers visible in metadata:", len(md.brokers))
print("Existing topics:")
for t in sorted(md.topics):
    print("  -", t)

Line by line

from confluent_kafka.admin import AdminClient
confluent_kafka is the client library you installed. It has three main client types: a Producer (writes events), a Consumer (reads events), and an AdminClient (manages/inspects the cluster — create topics, list topics, read metadata). Here we only want to inspect the cluster, so we import the admin one. Think of AdminClient as the "control panel" client, not a data client.

admin = AdminClient(conf)
This creates an admin client object, handing it your conf dictionary — the bootstrap server + SASL_SSL credentials from Cell 2. Important: creating the object doesn't connect yet. It just holds the configuration, ready to connect when you first ask it to do something. (Kafka clients connect lazily — on the first real call.)

md = admin.list_topics(timeout=10)
This is the line that actually talks to Confluent. Despite the name "list_topics," it does more than topics — it fetches the full cluster metadata. This is exactly the bootstrap flow we discussed: the client contacts the bootstrap server, and the cluster replies with the directory — the list of brokers and their addresses, and every topic with its partitions and which broker leads each. The timeout=10 means "wait up to 10 seconds for that reply, then give up" (so a bad config fails cleanly instead of hanging forever). The result is stored in md — a metadata object holding everything the cluster told us.

print("Connected to the cluster.")
If the previous line returned without throwing an error, the metadata came back, which means the connection + authentication worked. So reaching this line is the proof of a successful connection. (If the config were wrong, list_topics would have raised an exception and we'd never get here.)

print("Brokers visible in metadata:", len(md.brokers))
md.brokers is the broker directory the cluster sent back — a collection of the brokers the client can see. len(...) counts them. This is the literal answer to "how many brokers did the bootstrap server tell me about?" — the front-desk directory, as a number.

for t in sorted(md.topics):
md.topics is the collection of topics the cluster reported (a dictionary keyed by topic name). sorted(...) just alphabetizes the names so the output is tidy. We loop over each topic name t.

print(" -", t)
Prints each topic name, indented with " -" for readability. This is why you saw lab-events show up — it's an existing topic the cluster reported in the metadata.

The one idea to take from this

The whole cell is a concrete demonstration of the bootstrap → metadata handshake from our architecture talk. You gave the client one address (the bootstrap server), and in a single call it learned the entire cluster — all brokers and all topics.

In [0]:
%skip
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient(conf)

new_topic = NewTopic(
    topic="post-engagement",
    num_partitions=3,
    replication_factor=3,     # Confluent Cloud manages/fixes this at 3; each partition copied on 3 brokers
)

futures = admin.create_topics([new_topic])   # returns a dict: {topic_name: Future}

for name, f in futures.items():
    try:
        f.result()           # blocks until the broker finishes; raises if it failed
        print(f"Created topic: {name}")
    except Exception as e:
        print(f"Could not create {name}: {e}")

In [0]:
md = admin.list_topics(timeout=10)
t = md.topics["post-engagement"]
print("Topic:", "post-engagement")
print("Partitions:", len(t.partitions))
for pid, p in sorted(t.partitions.items()):
    print(f"  partition {pid}: leader broker id = {p.leader}, replicas = {p.replicas}")

In [0]:
type(t)

In [0]:
print(t.topic)

In [0]:
type(t.partitions)

In [0]:
print(t.partitions)

In [0]:
type(t.partitions[0])

In [0]:
for i in range(0,len(t.partitions)):
    print(t.partitions[i].replicas)
